# Capitolo 15 – Ricorrenze: RecursiveSum, Torre di Hanoi e forme chiuse (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap15_ricorrenze_cpp.ipynb)

Verifica numerica degli esempi del capitolo: l'algoritmo **RecursiveSum** con il conteggio dei passi $S(n)$ (che soddisfa $S(1)=2$, $S(n)=S(n-1)+3$, cioè $S(n)=3n-1$), la **Torre di Hanoi** che conta le mosse ($2^n-1$) e il confronto fra alcune ricorrenze del capitolo, calcolate direttamente, e la loro forma chiusa.

Il capitolo completo, con pseudocodice, dimostrazioni ed esempi, è [sul sito](https://fabiofurini.github.io/algoritmi/analisi-ricorsione/15-ricorrenze/).

Ogni programma è scritto in un file con la cella `%%writefile` e poi compilato ed eseguito con `g++ -std=c++17 -O2`: si eseguono le celle in ordine, dall'alto in basso. Nessuna libreria esterna.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `RecursiveSum` → `RecursiveSum` (PascalCase).

## 1. RecursiveSum e il conteggio dei passi

**RecursiveSum**$(a,n)$ — *Input*: un array $a[1..n]$ di $n\ge1$ interi. *Output*: la somma $a[1]+\dots+a[n]$.

Il codice è fedele allo pseudocodice (righe 2, 3, 5, 6); un contatore `steps` somma i passi eseguiti da ogni chiamata, come nel capitolo: 2 passi se $n=1$, 3 passi più la chiamata ricorsiva se $n>1$. La seconda funzione è `sumRecDecrease` del codice del corso, che si ferma all'array vuoto ($n=0$) e quindi esegue una chiamata in più.

In [1]:
%%writefile recursive_sum.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long steps = 0;   // passi eseguiti (righe di pseudocodice)
int calls = 0;         // numero di chiamate

// Somma di a[0..n-1] (pseudocodice: a[1..n]), n >= 1
int RecursiveSum(int *a, int n)
{
    calls++;
    steps++;                  // riga 2: if n = 1
    if (n == 1)
    {
        steps++;              // riga 3: return a[1]
        return a[0];
    }
    steps += 2;               // righe 5 e 6
    int s = RecursiveSum(a, n - 1);
    return s + a[n - 1];
}

// Variante del codice del corso: caso base n = 0
long long sumRecDecrease(int *a, int n)
{
    calls++;
    if (n == 0)
    {
        return 0;
    }
    return sumRecDecrease(a, n - 1) + a[n - 1];
}

#ifndef NOMAIN
int main()
{
    int a[] = {3, 1, 4, 1, 5, 9, 2, 6};
    int n = 8;
    printf("a = [3, 1, 4, 1, 5, 9, 2, 6]\n");
    calls = 0;
    int r = RecursiveSum(a, n);
    printf("RecursiveSum(a, 8) = %d con %d chiamate\n", r, calls);
    calls = 0;
    long long r2 = sumRecDecrease(a, n);
    printf("sumRecDecrease(a, 8) = %lld con %d chiamate\n", r2, calls);
    printf("n  S(n) misurato  3n-1\n");
    for (int k = 1; k <= 8; k++)
    {
        steps = 0;
        RecursiveSum(a, k);
        printf("%-2d %12lld %5d\n", k, steps, 3 * k - 1);
    }
    return 0;
}
#endif

Writing recursive_sum.cpp


In [2]:
!g++ -std=c++17 -O2 -o recursive_sum recursive_sum.cpp && ./recursive_sum

a = [3, 1, 4, 1, 5, 9, 2, 6]
RecursiveSum(a, 8) = 31 con 8 chiamate
sumRecDecrease(a, 8) = 31 con 9 chiamate
n  S(n) misurato  3n-1
1             2     2
2             5     5
3             8     8
4            11    11
5            14    14
6            17    17
7            20    20
8            23    23


**Prova tu.** Cambia l'array `a` (anche la lunghezza) e verifica che i passi misurati siano sempre $3n-1$.

In [3]:
%%writefile prova_recursive_sum.cpp
#define NOMAIN
#include "recursive_sum.cpp"

int main()
{
    // Cambia l'array e controlla che i passi misurati siano sempre 3n-1.
    int a[] = {7, -2, 5, 0, 11};
    int n = 5;
    steps = 0;
    int r = RecursiveSum(a, n);
    printf("%d %lld %d\n", r, steps, 3 * n - 1);
    return 0;
}

Writing prova_recursive_sum.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_recursive_sum prova_recursive_sum.cpp && ./prova_recursive_sum

21 14 14


## 2. Torre di Hanoi: $T(n)=2T(n-1)+1$

**Hanoi**$(n, from, to, aux)$ — *Input*: il numero di dischi $n\ge1$ e i tre pioli. *Output*: la sequenza di mosse che trasferisce i dischi $1,\dots,n$ da *from* a *to*.

Il codice conta le mosse (e, per $n=3$, le stampa): il capitolo mostra che sono esattamente $2^n-1$. La chiamata iniziale è `Hanoi(n, 'A', 'C', 'B')`.

In [5]:
%%writefile hanoi.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long moves = 0;    // numero di mosse
bool printMoves = false;

void Hanoi(int n, char from, char to, char aux)
{
    if (n == 1)
    {
        moves++;                                // sposta il disco 1
        if (printMoves) printf("  disco 1: %c -> %c\n", from, to);
        return;
    }
    Hanoi(n - 1, from, aux, to);
    moves++;                                    // sposta il disco n
    if (printMoves) printf("  disco %d: %c -> %c\n", n, from, to);
    Hanoi(n - 1, aux, to, from);
}

#ifndef NOMAIN
int main()
{
    printf("Mosse per n = 3:\n");
    printMoves = true;
    Hanoi(3, 'A', 'C', 'B');
    printMoves = false;
    printf("n   mosse  2^n-1\n");
    int ns[] = {1, 2, 3, 4, 5, 6, 7, 8, 10};
    for (int n : ns)
    {
        moves = 0;
        Hanoi(n, 'A', 'C', 'B');
        printf("%-3d %5lld %5lld\n", n, moves, (1LL << n) - 1);
    }
    return 0;
}
#endif

Writing hanoi.cpp


In [6]:
!g++ -std=c++17 -O2 -o hanoi hanoi.cpp && ./hanoi

Mosse per n = 3:
  disco 1: A -> C
  disco 2: A -> B
  disco 1: C -> B
  disco 3: A -> C
  disco 1: B -> A
  disco 2: B -> C
  disco 1: A -> C
n   mosse  2^n-1
1       1     1
2       3     3
3       7     7
4      15    15
5      31    31
6      63    63
7     127   127
8     255   255
10   1023  1023


**Prova tu.** Cambia il numero di dischi $n$: le mosse crescono come $2^n-1$ (con $n=15$ sono già $32\,767$).

In [7]:
%%writefile prova_hanoi.cpp
#define NOMAIN
#include "hanoi.cpp"

int main()
{
    // Prova con più dischi: quante mosse servono per n = 15?
    int n = 15;
    moves = 0;
    Hanoi(n, 'A', 'C', 'B');
    printf("%d %lld %lld\n", n, moves, (1LL << n) - 1);
    return 0;
}

Writing prova_hanoi.cpp


In [8]:
!g++ -std=c++17 -O2 -o prova_hanoi prova_hanoi.cpp && ./prova_hanoi

15 32767 32767


## 3. Ricorrenze calcolate direttamente e forme chiuse

Ogni ricorrenza è calcolata con una funzione ricorsiva che segue la definizione e confrontata con la forma chiusa ottenuta nel capitolo (metodo dell'espansione). Per le ricorrenze con $n/2$ i valori di $n$ sono potenze di 2.

| ricorrenza | forma chiusa |
|---|---|
| $T(n)=T(n-1)+n$, $T(1)=1$ | $n(n+1)/2$ |
| $S(n)=S(n-1)+3$, $S(1)=2$ (RecursiveSum) | $3n-1$ |
| $T(n)=2T(n-1)+1$, $T(1)=1$ (Hanoi) | $2^n-1$ |
| $T(n)=T(n/2)+1$, $T(1)=1$ | $\log_2 n+1$ |
| $T(n)=2T(n/2)+n$, $T(1)=1$ | $n\log_2 n+n$ |

In [9]:
%%writefile recurrences.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long TQuad(long long n)  { return n == 1 ? 1 : TQuad(n - 1) + n; }
long long TSum(long long n)   { return n == 1 ? 2 : TSum(n - 1) + 3; }
long long THanoi(long long n) { return n == 1 ? 1 : 2 * THanoi(n - 1) + 1; }
long long TLog(long long n)   { return n == 1 ? 1 : TLog(n / 2) + 1; }
long long TNLogN(long long n) { return n == 1 ? 1 : 2 * TNLogN(n / 2) + n; }

long long Log2(long long n)   // n è una potenza di 2
{
    return 63 - __builtin_clzll(n);
}

void Confronta(const string &nome, vector<long long> valori,
               function<long long(long long)> ric, function<long long(long long)> chiusa)
{
    printf("%s\n", nome.c_str());
    for (long long n : valori)
    {
        long long r = ric(n), c = chiusa(n);
        printf("  n=%-7lld %12lld %12lld  %s\n", n, r, c, r == c ? "OK" : "DIVERSI");
    }
}

#ifndef NOMAIN
int main()
{
    Confronta("T(n) = T(n-1) + n  ->  n(n+1)/2", {1, 2, 3, 4, 10, 100}, TQuad, [](long long n) { return n * (n + 1) / 2; });
    Confronta("S(n) = S(n-1) + 3  ->  3n-1", {1, 2, 3, 4, 100}, TSum, [](long long n) { return 3 * n - 1; });
    Confronta("T(n) = 2T(n-1) + 1  ->  2^n-1", {1, 2, 3, 4, 10, 20}, THanoi, [](long long n) { return (1LL << n) - 1; });
    Confronta("T(n) = T(n/2) + 1  ->  log2(n)+1", {1, 2, 4, 8, 1024, 1 << 20}, TLog, [](long long n) { return Log2(n) + 1; });
    Confronta("T(n) = 2T(n/2) + n  ->  n log2(n) + n", {1, 2, 4, 8, 1024, 1 << 20}, TNLogN, [](long long n) { return n * Log2(n) + n; });
    return 0;
}
#endif

Writing recurrences.cpp


In [10]:
!g++ -std=c++17 -O2 -o recurrences recurrences.cpp && ./recurrences

T(n) = T(n-1) + n  ->  n(n+1)/2
  n=1                  1            1  OK
  n=2                  3            3  OK
  n=3                  6            6  OK
  n=4                 10           10  OK
  n=10                55           55  OK
  n=100             5050         5050  OK
S(n) = S(n-1) + 3  ->  3n-1
  n=1                  2            2  OK
  n=2                  5            5  OK
  n=3                  8            8  OK
  n=4                 11           11  OK
  n=100              299          299  OK
T(n) = 2T(n-1) + 1  ->  2^n-1
  n=1                  1            1  OK
  n=2                  3            3  OK
  n=3                  7            7  OK
  n=4                 15           15  OK
  n=10              1023         1023  OK
  n=20           1048575      1048575  OK
T(n) = T(n/2) + 1  ->  log2(n)+1
  n=1                  1            1  OK
  n=2                  2            2  OK
  n=4                  3            3  OK
  n=8                  4            4

**Prova tu.** Aggiungi una ricorrenza tua (qui $T(n)=T(n-1)+2n-1$, $T(1)=1$) e confrontala con la sua forma chiusa.

In [11]:
%%writefile prova_recurrences.cpp
#define NOMAIN
#include "recurrences.cpp"

int main()
{
    // Aggiungi una ricorrenza: T(n) = T(n-1) + 2n - 1 con T(1) = 1 ha forma chiusa n^2.
    function<long long(long long)> tMia = [&](long long n) -> long long { return n == 1 ? 1 : tMia(n - 1) + 2 * n - 1; };
    Confronta("T(n) = T(n-1) + 2n - 1  ->  n^2", {1, 2, 3, 10, 100}, tMia, [](long long n) { return n * n; });
    return 0;
}

Writing prova_recurrences.cpp


In [12]:
!g++ -std=c++17 -O2 -o prova_recurrences prova_recurrences.cpp && ./prova_recurrences

T(n) = T(n-1) + 2n - 1  ->  n^2
  n=1                  1            1  OK
  n=2                  4            4  OK
  n=3                  9            9  OK
  n=10               100          100  OK
  n=100            10000        10000  OK
